<a href="https://colab.research.google.com/github/TuanKiet04/RielFezzTool/blob/main/LLM_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Đánh giá LLM

## Mục tiêu
* Đánh giá hiệu suất các mô hình sau:
  * GPT
  * Gemini
  * Mistral
  * Llama
  * Gemma
* Kết luận được mô hình phù hợp cho hệ thống Crawler
* Bên cạnh đó đánh giá phương pháp lọc đầu vào để giảm input token

## Chuẩn bị tập test

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
urls = [
    'https://vnexpress.net/khong-quan-ukraine-tuyen-bo-ban-ha-tiem-kich-su-35-nga-4895862.html',
    'https://dantri.com.vn/xa-hoi/bo-chinh-tri-sap-nhap-tinh-dong-thoi-voi-hop-nhat-xa-o-noi-du-dieu-kien-20250607170102455.htm',
    'https://thanhnien.vn/dang-nha-nuoc-luon-quan-tam-cong-dong-nguoi-viet-o-nuoc-ngoai-185250607171121924.htm',
]

raw_htmls = []
for url in urls:
  response = requests.get(url)
  soup = BeautifulSoup(response.content, 'html.parser')
  html = soup.prettify()
  raw_htmls.append(html)

## Chuẩn bị mô hình

In [ ]:
import json
import time
from openai import OpenAI
from google import genai

open_router_api_key = 'enter_your_key_api_here'
google_api_key = 'enter_your_key_api_here'

In [ ]:
models = {
    "openai/gpt-oss-20b:free": "openai/gpt-oss-20b",
    "google/gemma-3-27b-it:free": "google/gemma-3-27b-it",
    "gemini-2.0-flash-lite-001": "gemini-2.0-flash-lite",
    "gemini-2.0-flash-001": "gemini-2.0-flash",
    "gemini-2.5-flash-preview-05-20": "gemini-2.5-flash-preview",
}

model_names = {
    "openai/gpt-oss-20b:free": "openai/GPT-oss-20b:free",
    "google/gemma-3-27b-it:free": "Gemma-3-27b",
    "gemini-2.0-flash-lite-001": "Gemini-2.0-flash-lite",
    "gemini-2.0-flash-001": "Gemini-2.0-flash",
    "gemini-2.5-flash-preview-05-20": "Gemini-2.5-flash-preview",
}

In [ ]:
def build_prompt(html: str):
  return f"""Extract the requirements below from the HTML with absolute precision.

HTML: {html}

EXTRACTION REQUIREMENTS:
1. TITLE: Extract the main article headline/title exactly as displayed
2. CREATOR: Extract author
3. CONTENT: Extract the complete article body text maintaining original structure

CONTENT EXTRACTION RULES:
CRITICAL: Extract content EXACTLY as written - NO modifications allowed
- Preserve ALL original paragraphs in their exact order
- Maintain original paragraph breaks and formatting
- Keep anything related to the content of the article, don't be distracted by the '-', this may be contains the conversation of the newspaperman.
- Do NOT summarize, paraphrase, condense, or rewrite ANY text
- Do NOT combine or split paragraphs
- Do NOT correct grammar, spelling, or punctuation

Return ONLY a valid JSON object with this exact structure (no markdown, no code blocks, no additional text):
{{
  "title": "extracted title here",
  "creator": "extracted author or Staff",
  "content": "complete article content here"
}}
"""

In [ ]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=open_router_api_key,
)

gemini_client = genai.Client(api_key=google_api_key)

In [ ]:
import regex

def extract_outermost_json_block(text: str) -> str:
    match = regex.search(r'\{(?:[^{}]|(?R))*\}', text.strip())
    return match.group(0) if match else ''

In [ ]:
def call_model(model, prompt):
    start_time = time.perf_counter()
    json_str = ""
    raw_response = ""

    try:
        if model.startswith("gemini-"):
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
            )
            raw_response = response.text
        else:
            completion = client.chat.completions.create(
                extra_body={},
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
            raw_response = completion.choices[0].message.content

        # In ra để debug
        print(f"Raw response length: {len(raw_response)}")
        print(f"First 200 chars: {raw_response[:200]}")

        # Làm sạch response
        # Loại bỏ markdown code blocks nếu có
        if "```json" in raw_response:
            json_str = raw_response.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_response:
            json_str = raw_response.split("```")[1].split("```")[0].strip()
        else:
            json_str = raw_response.strip()

        # Thử extract JSON nếu vẫn không được
        if not json_str.startswith("{"):
            json_str = extract_outermost_json_block(json_str)

        end_time = time.perf_counter()
        elapsed_time = end_time - start_time

        print(f"Extracted JSON length: {len(json_str)}")

        result = json.loads(json_str)
        return {
            **result,
            "time_res": f"{round(elapsed_time * 1000)}ms"
        }

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from model {model}: {e}")
        print(f"Raw response: {raw_response[:500]}")
        print(f"Extracted string: {json_str[:500]}")
        return {
            "error": f"JSON Decode Error: {e}",
            "title": "ERROR",
            "creator": "ERROR",
            "content": "ERROR",
            "time_res": f"{round((time.perf_counter() - start_time) * 1000)}ms"
        }
    except Exception as e:
        print(f"An unexpected error occurred with model {model}: {e}")
        return {
            "error": f"Unexpected Error: {e}",
            "title": "ERROR",
            "creator": "ERROR",
            "content": "ERROR",
            "time_res": f"{round((time.perf_counter() - start_time) * 1000)}ms"
        }

In [ ]:
def compare_models(html):
  prompt = build_prompt(html)
  results = {}
  for model in models.keys():
    print(f"Querying {model}...")
    output = call_model(model, prompt)
    results[model] = output
  return results

## Kiểm thử

### Tối ưu input token bằng cách lọc raw HTML, giữ lại các thẻ cần thiết

In [ ]:
# filter
filtered_htmls = []
remove_elements = [
    "script", "noscript", "style", "head", "footer", "nav", "[class*='menu']", "[class*='nav']",
    "svg", "img", "link", ".ads", ".advertisment", ".social", "[class*='banner']",
    "[id*='ads']", "[id*='banner']", "[class*='share']", "[id*='share']", "[class*='comment']",
    "[id*='comment']", ".copy-right", "#copy-right", "header", "footer", "aside", ".sidebar", ".navigation", ".menu", ".social-share", ".comments",
    ".script", ".style", ".ads", ".advertisement", ".related-posts", "form", "button", ".breadcrumb", ".img", ".javascript", "#copy-right", ".copy-right"
]

for html in raw_htmls:
    soup = BeautifulSoup(html, "html.parser")

    # xóa các selector thông thường
    for selector in remove_elements:
        for tag in soup.select(selector):
            tag.decompose()

    # xóa <header> nếu không có <h1> bên trong
    for header_tag in soup.find_all("header"):
        if not header_tag.find("h1"):
            header_tag.decompose()

    filtered_htmls.append(soup.prettify())


In [ ]:
import tiktoken
from transformers import AutoTokenizer
import pandas as pd
from IPython.display import clear_output

In [ ]:
# create HF_TOKEN in Colab secrets to use AutoTokenizer
from huggingface_hub import login
hf_token ='enter_your_key_api_here'
login(token=hf_token)

In [ ]:
def count_tokens(model, content):
  try:
      tokenizer = AutoTokenizer.from_pretrained(model)
      return len(tokenizer.encode(content))
  except Exception:
    print(f"Warning: {model} not found. Using cl100k_base encoding.")
    return len(tiktoken.get_encoding("cl100k_base").encode(content))

In [ ]:
tokens_before = pd.DataFrame({"Model": models.values()})
tokens_before.index += 1
tokens_before.index.name = "STT"

for i, raw_html in enumerate(raw_htmls, start=1):
    filtered_html = filtered_htmls[i-1]
    token_pairs = [
        count_tokens(model, raw_html)
        for model in models.values()
    ]
    tokens_before[f"Content {i}"] = token_pairs

clear_output(wait=True)
print("Tokens of content before filtering")
print("-----------------")
display(tokens_before)


Tokens of content before filtering
-----------------


,Model,Content 1,Content 2,Content 3
STT,,,,
1,openai/gpt-oss-20b,81805,152062,218229
2,meta-llama/Llama-3.3-70B-Instruct,81219,153976,217291
3,google/gemma-3-27b-it,81219,153976,217291
4,gemini-2.0-flash-lite,81219,153976,217291
5,gemini-2.0-flash,81219,153976,217291
6,gemini-2.5-flash-preview,81219,153976,217291


In [ ]:
tokens_after = pd.DataFrame({"Model": models.values()})
tokens_after.index += 1
tokens_after.index.name = "STT"

for i, filtered_html in enumerate(filtered_htmls, start=1):
    token_pairs = [
        count_tokens(model, filtered_html)
        for model in models.values()
    ]
    tokens_after[f"Content {i}"] = token_pairs

clear_output(wait=True)
print("Tokens of content after filtering")
print("-----------------")
display(tokens_after)


Tokens of content after filtering
-----------------


,Model,Content 1,Content 2,Content 3
STT,,,,
1,openai/gpt-oss-20b,3229,3122,7182
2,meta-llama/Llama-3.3-70B-Instruct,3587,4218,8278
3,google/gemma-3-27b-it,3587,4218,8278
4,gemini-2.0-flash-lite,3587,4218,8278
5,gemini-2.0-flash,3587,4218,8278
6,gemini-2.5-flash-preview,3587,4218,8278


In [ ]:
token_reduction_percent = pd.DataFrame({"Model": models.values()})
token_reduction_percent.index += 1
token_reduction_percent.index.name = "STT"

# ((before - after) / before) * 100
for col in tokens_before.columns[1:]:
    before = tokens_before[col]
    after = tokens_after[col]
    reduction_percent = ((before - after) / before * 100).round(2)
    token_reduction_percent[col] = reduction_percent.astype(str) + '%'

clear_output(wait=True)
print("Token reduction percentage after filtering")
print("------------------------------------------")
display(token_reduction_percent)


Token reduction percentage after filtering
------------------------------------------


,Model,Content 1,Content 2,Content 3
STT,,,,
1,openai/gpt-oss-20b,96.05%,97.95%,96.71%
2,meta-llama/Llama-3.3-70B-Instruct,95.58%,97.26%,96.19%
3,google/gemma-3-27b-it,95.58%,97.26%,96.19%
4,gemini-2.0-flash-lite,95.58%,97.26%,96.19%
5,gemini-2.0-flash,95.58%,97.26%,96.19%
6,gemini-2.5-flash-preview,95.58%,97.26%,96.19%


### Thực hiện kiểm thử các mô hình

In [ ]:
# urls = [
#     'https://vnexpress.net/khong-quan-ukraine-tuyen-bo-ban-ha-tiem-kich-su-35-nga-4895862.html',
#     'https://dantri.com.vn/xa-hoi/bo-chinh-tri-sap-nhap-tinh-dong-thoi-voi-hop-nhat-xa-o-noi-du-dieu-kien-20250607170102455.htm',
#     'https://thanhnien.vn/dang-nha-nuoc-luon-quan-tam-cong-dong-nguoi-viet-o-nuoc-ngoai-185250607171121924.htm',
# ]
expect_results = [
    {
      "title": "Không quân Ukraine tuyên bố bắn hạ tiêm kích Su-35 Nga",
      "creator": "Thùy Lâm",
      "content": '''
      "Quân đội Ukraine tuyên bố lực lượng không quân nước này đã bắn hạ một tiêm kích Su-35 của Nga tại tỉnh biên giới Kursk.
"Lực lượng không quân sáng 7/6 đã tiến hành chiến dịch thành công, bắn hạ một tiêm kích Su-35 Nga tại tỉnh Kursk", quân đội Ukraine hôm nay thông báo trên mạng xã hội, nhưng không cung cấp thêm thông tin chi tiết.

Không quân Ukraine sau đó công bố video một máy bay bị phá hủy, bốc cháy trên mặt đất, tuyên bố đây là tiêm kích Su-35 bị bắn hạ. Lực lượng này không cho biết đã sử dụng vũ khí nào để bắn rơi máy bay Nga.

Không quân Ukraine hiện trang bị tiêm kích F-16 do phương Tây viện trợ, cùng các dòng chiến đấu cơ cũ hơn từ thời Liên Xô.

Bộ Quốc phòng Nga chưa bình luận về thông tin, nhưng Pepel, báo địa phương ở tỉnh Belgorod của Nga, cho hay tiêm kích Su-35 đã rơi tại quận Kornevsky thuộc tỉnh Kursk. Fighterbomber, tài khoản mạng xã hội của một phi công quân sự Nga, xác nhận thông tin trên và cho hay phi công chiếc Su-35 đã phóng dù thoát hiểm sau khi máy bay bị bắn hạ.

Trước đó, Bộ Tổng tham mưu Ukraine cho biết Nga đã mất 413 máy bay chiến đấu kể từ đầu chiến sự. Lần gần nhất Ukraine tuyên bố bắn hạ tiêm kích Su-35 Nga bằng tên lửa phòng không là vào tháng 2/2024.

Nếu thông tin bắn hạ Su-35 được xác nhận, đòn tấn công này sẽ làm tăng thêm thiệt hại đối với không quân Nga trong tuần qua. Ukraine ngày 1/6 phát động chiến dịch Mạng nhện tấn công drone vào loạt căn cứ không quân nằm sâu trong lãnh thổ Nga, tuyên bố đánh trúng 41 máy bay và làm 20 chiếc bị hư hại.

Theo Kiev, đòn tập kích này đã gây thiệt hại khoảng 7 tỷ USD và vô hiệu hóa hơn 30% phi đội oanh tạc cơ chiến lược Nga. Trong khi đó, giới chức Mỹ ước tính chiến dịch Mạng nhện của Ukraine "đánh trúng không quá 20 máy bay Nga và phá hủy được khoảng 10 chiếc trong số đó"."
''',
    },
    {
      "title": "Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp nhất xã ở nơi đủ điều kiện",
      "creator": "Hoài Thu",
      "content": '''
      "(Dân trí) - Theo yêu cầu của Bộ Chính trị, Ban Bí thư, ban thường vụ các tỉnh ủy, thành ủy chủ động xem xét, quyết định và thời điểm đưa các cơ quan, đơn vị cấp tỉnh, cấp xã mới vào hoạt động ngay từ ngày 1/7.
Thường trực Ban Bí thư Trần Cẩm Tú vừa thay mặt Bộ Chính trị ký ban hành kết luận số 163 về thực hiện một số nội dung, nhiệm vụ khi sắp xếp tổ chức bộ máy và đơn vị hành chính.

Tại cuộc họp ngày 6/6, sau khi nghe báo cáo của Ban Tổ chức Trung ương về tình hình, tiến độ thực hiện các nghị quyết, kết luận của Trung ương, Bộ Chính trị về sắp xếp tổ chức bộ máy và đơn vị hành chính, Bộ Chính trị, Ban Bí thư thể hiện quan điểm cơ bản thống nhất.

Trong đó, Bộ Chính trị, Ban Bí thư thống nhất chủ trương tiến hành sắp xếp đơn vị hành chính cấp tỉnh đồng thời với thực hiện hợp nhất, sáp nhập đơn vị hành chính cấp xã đối với những địa phương đã chuẩn bị kỹ lưỡng, chu đáo, đầy đủ các điều kiện cần thiết và đã hoàn thiện phương án nhân sự cấp tỉnh, cấp xã.

Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp nhất xã ở nơi đủ điều kiện

Việc này cũng có thể tiến hành ở những địa phương sẵn sàng cơ sở vật chất, trang thiết bị, phương tiện để tổ chức bộ máy đi vào hoạt động ngay, bảo đảm thông suốt, đồng bộ và đã tổ chức vận hành thử nghiệm, rút kinh nghiệm về hoạt động của các cơ quan Đảng, Nhà nước, Mặt trận Tổ quốc, các đoàn thể cấp xã.

Bộ Chính trị, Ban Bí thư giao ban thường vụ các tỉnh ủy, thành ủy chủ động xem xét, quyết định thời điểm đưa các tổ chức, cơ quan, đơn vị cấp tỉnh, cấp xã mới vào hoạt động, có thể thực hiện ngay từ ngày 1/7, khi Hiến pháp năm 2013 (sửa đổi) có hiệu lực thi hành và cấp có thẩm quyền đã ban hành các quyết định có liên quan.

Ngoài ra, Bộ Chính trị, Ban Bí thư thống nhất chủ trương giao ban thường vụ các tỉnh ủy, thành ủy lãnh đạo, chỉ đạo việc quản lý tổng biên chế năm 2025, thực hiện tinh giản biên chế và cơ cấu lại đội ngũ cán bộ, công chức, viên chức theo đúng chủ trương, quy định của Đảng, pháp luật của Nhà nước.

Ban thường vụ các tỉnh ủy, thành ủy được chủ động quyết định việc điều chuyển số chỉ tiêu biên chế giữa khối chính quyền và khối Đảng, đoàn thể ở địa phương trong quá trình sắp xếp tổ chức bộ máy, đơn vị hành chính các cấp, thực hiện mô hình chính quyền địa phương 2 cấp.

Nhưng yêu cầu được Bộ Chính trị đặt ra là phải bảo đảm không thay đổi tổng biên chế và cơ cấu cán bộ, công chức, viên chức đã được Ban Tổ chức Trung ương giao cho địa phương, đồng thời kịp thời báo cáo Ban Tổ chức Trung ương số biên chế điều chuyển để tổng hợp, theo dõi.

Đảng ủy Chính phủ được giao lãnh đạo, chỉ đạo khẩn trương ban hành các nghị định về phân cấp, phân quyền, phân định thẩm quyền và hướng dẫn chức năng, nhiệm vụ, tổ chức bộ máy của cơ quan chuyên môn thuộc UBND cấp tỉnh, cấp xã và các văn bản khác có liên quan.

Trước mắt, Bộ Chính trị, Ban Bí thư cho biết có thể giữ nguyên số lượng tổ chức và số lượng cấp phó của các cơ quan, tổ chức khi thực hiện sáp nhập.

Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp nhất xã ở nơi đủ điều kiện - 2
Tổng Bí thư Tô Lâm chủ trì cuộc họp Bộ Chính trị cho ý kiến về tình hình, tiến độ thực hiện các nghị quyết, kết luận của Trung ương, Bộ Chính trị về sắp xếp tổ chức bộ máy và đơn vị hành chính ngày 6/6 (Ảnh: TTXVN).

Bộ Chính trị cũng nêu định hướng thống nhất tổ chức cơ quan chuyên môn ở các xã không sáp nhập, hợp nhất, đồng bộ với quy định về tổ chức bộ máy cơ quan tham mưu, giúp việc cấp ủy xã, phường, đặc khu theo quy định thi hành Điều lệ Đảng và phù hợp với chức năng, nhiệm vụ, thẩm quyền của chính quyền cấp xã mới.

Việc hướng dẫn giải quyết kịp thời chế độ, chính sách đối với cán bộ, công chức, viên chức, người lao động nghỉ công tác; nghiên cứu lộ trình kéo dài việc sử dụng người không chuyên trách phù hợp với thời điểm sắp xếp lại thôn, tổ dân phố cũng được Bộ Chính trị, Ban Bí thư lưu ý.

Bộ Chính trị, Ban Bí thư giao Đảng ủy Quốc hội lãnh đạo, chỉ đạo thông qua Hiến pháp, các luật, nghị quyết có liên quan; hướng dẫn về tổ chức và hoạt động của Đoàn đại biểu Quốc hội và HĐND các cấp sau sắp xếp.

Với các địa phương, Bộ Chính trị, Ban Bí thư lưu ý cần tập trung làm tốt công tác cán bộ, chính trị, tư tưởng, giải quyết kịp thời chế độ, chính sách đối với cán bộ, công chức, viên chức, người lao động nghỉ công tác; thực hiện nghiêm việc rà soát, thống kê, bàn giao tài sản, tài liệu, bảo đảm đầy đủ, đồng bộ, thông suốt trong quá trình chuyển giao.

Về công tác nhân sự, theo yêu cầu của Bộ Chính trị, trong thời gian cấp ủy cấp trên trực tiếp chưa chỉ định nhân sự cấp ủy khóa mới, cấp ủy (chi ủy) khóa cũ có trách nhiệm tiếp tục lãnh đạo, chỉ đạo, điều hành hoạt động của đảng bộ, chi bộ cho đến khi cấp ủy cấp trên trực tiếp chỉ định cấp ủy (chi ủy) khóa mới."
''',
    },
    {
      "title": "Đảng, Nhà nước luôn quan tâm cộng đồng người Việt ở nước ngoài",
      "creator": "Đậu Tiến Đạt",
      "content": '''
      "Sáng 7.6, tại thủ đô Tallinn, trong chương trình thăm chính thức Estonia, Thủ tướng Phạm Minh Chính cùng phu nhân Lê Thị Bích Trân đã gặp gỡ các cán bộ Đại sứ quán và cộng đồng người Việt Nam tại Estonia.
Đại sứ Việt Nam tại Phần Lan kiêm nhiệm Estonia Phạm Thị Thanh Bình cho biết, cộng đồng người Việt Nam tại Estonia lúc đông nhất có khoảng 200 người, hiện có khoảng 50 người, chủ yếu là trí thức, sinh viên, doanh nhân.

 - Ảnh 1.
Thủ tướng Phạm Minh Chính cùng phu nhân Lê Thị Bích Trân gặp gỡ cộng đồng người Việt Nam tại Estonia

ẢNH: NHẬT BẮC

Estonia là nước có trình độ số hóa hàng đầu thế giới, trong đó có chương trình công dân kỹ thuật số toàn cầu (e-Residency), hiện đã có 201 người Việt và 45 doanh nghiệp Việt Nam tham gia chương trình này.

Phát biểu tại cuộc gặp gỡ, Thủ tướng Phạm Minh Chính nhấn mạnh, chuyến thăm lần này sẽ mở ra chương mới trong hợp tác giữa hai nước, đặc biệt trong các lĩnh vực Estonia có thế mạnh và Việt Nam có nhu cầu như chính phủ điện tử, kinh tế số, trí tuệ nhân tạo, an ninh mạng, công nghệ tài chính.

Thủ tướng đánh giá cao tình cảm của cộng đồng người Việt tại Estonia, tuy chưa đông nhưng phần lớn là trí thức, giàu lòng yêu nước, luôn hướng về quê hương, luôn trân trọng tình cảm, sự quan tâm của Đảng, Nhà nước, tuân thủ pháp luật của nước sở tại và nỗ lực vươn lên và cơ bản thành công. Phía Estonia cũng rất trân trọng, đánh giá cao cộng đồng người Việt Nam tại đây.

Thủ tướng nhấn mạnh, với quan điểm không để ai lại phía sau trong quá trình phát triển, cộng đồng người Việt Nam ở nước ngoài là bộ phận không tách rời của cộng đồng các dân tộc Việt Nam, dù cộng đồng chỉ có một người thì Đảng, Nhà nước cũng quan tâm, hỗ trợ để mỗi người ổn định cuộc sống và đóng góp cho cộng đồng, cho đất nước, cũng như vun đắp cho quan hệ giữa Việt Nam và nước sở tại. Đây là quan điểm xuyên suốt trong quá trình bảo vệ, xây dựng, phát triển đất nước.

Thời gian qua, một số chính sách đối với người Việt Nam ở nước ngoài tiếp tục được cụ thể hóa vào các dự thảo luật, như luật Căn cước, luật Nhà ở, luật Đất đai. Chính phủ đang tiếp tục trình Quốc hội sửa đổi luật Quốc tịch, với tinh thần thông thoáng hơn.

Thủ tướng ghi nhận, đánh giá cao và đề nghị Đại sứ quán Việt Nam tại Phần Lan kiêm nhiệm Estonia tiếp tục nỗ lực thực hiện tốt các nhiệm vụ được giao, đặc biệt là trong quan tâm, chăm lo đời sống vật chất và tinh thần của kiều bào và thúc đẩy thực chất quan hệ hữu nghị, hợp tác giữa Việt Nam với Estonia.

Thủ tướng cũng đề nghị các cơ quan liên quan hỗ trợ bà con thúc đẩy kết nối giao thương, đầu tư, các hoạt động quảng bá, xuất nhập khẩu các mặt hàng thế mạnh trong nước.

Chỉ đạo các cơ quan tiếp tục nghiên cứu có chính sách visa, lao động, đi lại thuận lợi hơn cho bà con, Thủ tướng lưu ý, trong thời gian Việt Nam chưa lập đại sứ quán tại Estonia và Estonia chưa lập đại sứ quán tại Việt Nam, Bộ Ngoại giao cần thúc đẩy thành lập cơ quan lãnh sự danh dự của nước này ở nước kia để tạo thuận lợi cho giao lưu giữa nhân dân, doanh nghiệp hai nước.

Thủ tướng mong bà con tiếp tục đoàn kết, thống nhất, phát triển cộng đồng, phát huy "tình dân tộc, nghĩa đồng bào", người đi trước giúp người đi sau, nỗ lực, cố gắng vươn lên, khai thác tốt hệ sinh thái khởi nghiệp năng động của Estonia để phát triển kinh doanh; tiếp tục góp ý về thể chế, cơ chế, chính sách, góp phần để đất nước "bắt kịp, tiến cùng và vượt lên" trong quá trình hội nhập, phát huy vai trò dẫn dắt trong một số lĩnh vực, giống như cách Estonia đã làm."
''',
    },
]

In [ ]:
results_multi_model = []

for html in filtered_htmls:
    print(f"\n{'='*50}")
    print(f"Processing HTML {len(results_multi_model) + 1}/{len(filtered_htmls)}")
    print(f"{'='*50}")
    results_multi_model.append(compare_models(html))

clear_output(wait=True)
print(f"Completed! Processed {len(results_multi_model)} HTML documents")

Completed! Processed 3 HTML documents


In [ ]:
# Hiển thị kết quả so sánh
import pandas as pd
from IPython.display import display

fields = ["title", "creator", "content", "time_res"]

for i, exp_result in enumerate(expect_results, start=1):
    print(f"\n{'='*80}")
    print(f"CONTENT {i}: {urls[i-1]}")
    print(f"{'='*80}\n")

    # Tạo DataFrame để so sánh
    data = {
        "Field": fields,
        "Expected Result": [
            exp_result.get(f, "")[:100] + "..." if f == "content" and len(str(exp_result.get(f, ""))) > 100
            else exp_result.get(f, "") if f != "time_res"
            else ""
            for f in fields
        ]
    }

    # Thêm kết quả từ các model
    for model_name, result in results_multi_model[i-1].items():
        data[model_names[model_name]] = [
            result.get(f, "")[:100] + "..." if f == "content" and len(str(result.get(f, ""))) > 100
            else result.get(f, "")
            for f in fields
        ]

    df = pd.DataFrame(data)
    df.set_index("Field", inplace=True)
    display(df)


CONTENT 1: https://vnexpress.net/khong-quan-ukraine-tuyen-bo-ban-ha-tiem-kich-su-35-nga-4895862.html



,Expected Result,openai/GPT-oss-20b:free,Llama-3.3-70B,Gemma-3-27b,Gemini-2.0-flash-lite,Gemini-2.0-flash,Gemini-2.5-flash-preview
Field,,,,,,,
title,Không quân Ukraine tuyên bố bắn hạ tiêm kích S...,Không quân Ukraine tuyên bố bắn hạ tiêm kích S...,Không quân Ukraine tuyên bố bắn hạ tiêm kích S...,ERROR,Không quân Ukraine tuyên bố bắn hạ tiêm kích S...,Không quân Ukraine tuyên bố bắn hạ tiêm kích S...,Không quân Ukraine tuyên bố bắn hạ tiêm kích S...
creator,Thùy Lâm,Staff,Thùy Lâm,ERROR,Thùy Lâm,Thùy Lâm,Thùy Lâm
content,"\n ""Quân đội Ukraine tuyên bố lực lượng k...",Quân đội Ukraine tuyên bố lực lượng không quân...,Quân đội Ukraine tuyên bố lực lượng không quân...,ERROR,Quân đội Ukraine tuyên bố lực lượng không quân...,Quân đội Ukraine tuyên bố lực lượng không quân...,Quân đội Ukraine tuyên bố lực lượng không quân...
time_res,,98742ms,3464ms,56764ms,4117ms,3947ms,34080ms



CONTENT 2: https://dantri.com.vn/xa-hoi/bo-chinh-tri-sap-nhap-tinh-dong-thoi-voi-hop-nhat-xa-o-noi-du-dieu-kien-20250607170102455.htm



,Expected Result,openai/GPT-oss-20b:free,Llama-3.3-70B,Gemma-3-27b,Gemini-2.0-flash-lite,Gemini-2.0-flash,Gemini-2.5-flash-preview
Field,,,,,,,
title,Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp ...,Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp ...,Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp ...,Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp ...,Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp ...,Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp ...,Bộ Chính trị: Sáp nhập tỉnh đồng thời với hợp ...
creator,Hoài Thu,Hoài Thu,Hoài Thu,Hoài Thu,Hoài Thu,Hoài Thu,Hoài Thu
content,"\n ""(Dân trí) - Theo yêu cầu của Bộ Chính...","(Dân trí) - Theo yêu cầu của Bộ Chính trị, Ban...","(Dân trí) - Theo yêu cầu của Bộ Chính trị, Ban...",Thường trực Ban Bí thư Trần Cẩm Tú vừa thay mặ...,"(Dân trí) - Theo yêu cầu của Bộ Chính trị, Ban...",Thường trực Ban Bí thư Trần Cẩm Tú vừa thay mặ...,"(Dân trí) - Theo yêu cầu của Bộ Chính trị, Ban..."
time_res,,24647ms,4861ms,20580ms,7622ms,6570ms,30292ms



CONTENT 3: https://thanhnien.vn/dang-nha-nuoc-luon-quan-tam-cong-dong-nguoi-viet-o-nuoc-ngoai-185250607171121924.htm



,Expected Result,openai/GPT-oss-20b:free,Llama-3.3-70B,Gemma-3-27b,Gemini-2.0-flash-lite,Gemini-2.0-flash,Gemini-2.5-flash-preview
Field,,,,,,,
title,"Đảng, Nhà nước luôn quan tâm cộng đồng người V...","Đảng, Nhà nước luôn quan tâm cộng đồng người V...",ERROR,"Đảng, Nhà nước luôn quan tâm cộng đồng người V...","Đảng, Nhà nước luôn quan tâm cộng đồng người V...","Đảng, Nhà nước luôn quan tâm cộng đồng người V...","Đảng, Nhà nước luôn quan tâm cộng đồng người V..."
creator,Đậu Tiến Đạt,Đậu Tiến Đạt,ERROR,Đậu Tiến Đạt,Đậu Tiến Đạt,Đậu Tiến Đạt,Đậu Tiến Đạt
content,"\n ""Sáng 7.6, tại thủ đô Tallinn, trong c...",Đại sứ Việt Nam tại Phần Lan kiêm nhiệm Estoni...,ERROR,"Sáng 7.6, tại thủ đô Tallinn, trong chương trì...",Đại sứ Việt Nam tại Phần Lan kiêm nhiệm Estoni...,Đại sứ Việt Nam tại Phần Lan kiêm nhiệm\n ...,"Sáng 7.6, tại thủ đô Tallinn, trong chương trì..."
time_res,,108781ms,8449ms,24694ms,5770ms,7398ms,17476ms


## Đánh giá & Kết luận

Kết quả đa số đều ở mức khá tốt > 80% nội dung không cần thiết bị loại bỏ đi. Đối với một số trang web có cấu trúc đặc biệt hơn thì vẫn đảm bảo loại bỏ được hơn 50% nội dung không cần thiết, đảm bảo được token đầu vào ở mức ~ 15000 hoặc ít hơn, dùng được cho các model thử nghiệm trong bài này.

### Đối với Kết quả Content Extractor thu được của các model

| Model                        | Đánh giá chung                          |  Thời gian phản hồi |
|-----------------------------|--------------------------------|--------------------------------|
| openai/gpt-oss-20b:free    | Chính xác 95% | 9-12s               |
| Llama-3.3-70B               | Chính xác 85% | 4–6s               |
| Gemini-2.0-flash-lite       | Chính xác 87%   | 3-5s                |
| Gemini-2.0-flash            | Chính xác 88%  | 5-7s                |
| Gemini-2.5-flash-preview    | Chính xác 90%       | 4–6s               |
